In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import argparse
import scanpy as sc
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter
import anndata as ad
import pandas as pd
from collections import Counter
import anndata
from sklearn.model_selection import train_test_split

from lib_vqvae.vqvae import VQVAE
from lib_vqvae.dataset import *

#### Load your h5ad data

In [2]:
# change to your file path
adata = sc.read_h5ad("../data/blood/blood.h5ad")
adata

AnnData object with n_obs × n_vars = 4755 × 33244
    obs: 'donor_id', 'author_cell_type', 'days_post_intubation', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'organism_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage'
    var: 'gene_symbols', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype'
    uns: 'default_embedding', 'schema_version', 'title'
    obsm: 'X_umap'

#### Load the pre-trained cell tokenizer

In [3]:
import warnings
warnings.filterwarnings("ignore")
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("--train_batch_size", type=int, default=64)
parser.add_argument("--eval_batch_size", type=int, default=2000)
parser.add_argument("--encoder_hidden_dim", type=list, default=[1600,1024,800])
parser.add_argument("--decoder_hidden_dim", type=list, default=[800,1024,1600])
parser.add_argument("--codebook_dim", type=int, default=8)
parser.add_argument("--n", type=int, default=32)
parser.add_argument("--n_codebooks", type=int, default=256)
parser.add_argument("--gamma", type=float, default=5)
parser.add_argument("--beta", type=float, default=10)
parser.add_argument('--weight_decay', type=float, default=5e-4)
parser.add_argument("--lr", type=float, default=1e-4) # 1e-3
parser.add_argument('--max_iter', type=int, default=20)
parser.add_argument('--seed', type=int, default=2024)
parser.add_argument('--gpu', type=int, default=0)
parser.add_argument("--dropout_rate", type=float, default=0.2)
parser.add_argument('--mode', type=str, default="vqvae")
parser.add_argument('--hvg', type=int, default=5000)

args = parser.parse_args([])

# -- device
if torch.cuda.is_available():
    args.device='cuda'
    torch.cuda.set_device(args.gpu)
else:
    args.device='cpu' 
print(args.device)
    
data_dim = 5000
vae = VQVAE(data_dim, args.n, args.codebook_dim, args.n_codebooks, args.encoder_hidden_dim, args.decoder_hidden_dim, args.beta, args.gamma, args.dropout_rate).to(args.device)

vae.load_state_dict(torch.load('../model/cell_tokenzier/model.pth')['model'])
vae.eval() 

cuda


VQVAE(
  (encoder): Encoder(
    (network): Sequential(
      (0): Linear(in_features=5000, out_features=1600, bias=True)
      (1): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
      (2): PReLU(num_parameters=1)
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=1600, out_features=1024, bias=True)
      (5): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (6): PReLU(num_parameters=1)
      (7): Dropout(p=0.2, inplace=False)
      (8): Linear(in_features=1024, out_features=800, bias=True)
      (9): LayerNorm((800,), eps=1e-05, elementwise_affine=True)
      (10): PReLU(num_parameters=1)
      (11): Dropout(p=0.2, inplace=False)
      (12): Linear(in_features=800, out_features=256, bias=True)
    )
  )
  (vector_quantization): VectorQuantizer(
    (embedding): Embedding(256, 8)
  )
  (decoder): Decoder(
    (network): Sequential(
      (0): Linear(in_features=256, out_features=800, bias=True)
      (1): LayerNorm((800,), eps=1e-05, elementwise

#### Map adata to the selected 5000 gene set

In [4]:
var_names_df = pd.read_csv("../data/5000_gene_names.csv")
gene_names = var_names_df['gene'].tolist()

existing_genes = adata.var["feature_name"].isin(gene_names)
adata_filtered = adata[:, existing_genes]

existing_gene_names = adata.var['feature_name'][existing_genes].tolist()
result_df = pd.DataFrame(index=adata.obs_names, columns=gene_names)
result_df[:] = 0

if issparse(adata_filtered.X):
    adata_filtered_df = adata_filtered.X.toarray()
else:
    adata_filtered_df = adata_filtered.X

result_df.loc[:, existing_gene_names] = adata_filtered_df

new_adata = sc.AnnData(X=result_df.values.astype(np.float32), obs=adata.obs, var=pd.DataFrame(index=gene_names))
new_adata.var['feature_name'] = gene_names
new_adata.var_names = gene_names

new_adata

AnnData object with n_obs × n_vars = 4755 × 5000
    obs: 'donor_id', 'author_cell_type', 'days_post_intubation', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'organism_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage'
    var: 'feature_name'

#### Process the data

In [5]:
adata = new_adata
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

#### Get the codebooks

In [6]:
vae.get_adata_codebook_index(adata, mode=args.mode, device=args.device)

AnnData object with n_obs × n_vars = 4755 × 5000
    obs: 'donor_id', 'author_cell_type', 'days_post_intubation', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'organism_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'suspension_type', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'batch'
    var: 'feature_name'
    uns: 'log1p'
    obsm: 'code_index', 'latent', 'recon_x'

In [7]:
del adata.X
del adata.obsm['recon_x']
del adata.obsm['latent']

#### Save the code data

In [13]:
adata.write_h5ad("../data/blood/blood_code.h5ad")